# Tutorial 21: YOLO26 Object Detection, Pose, Segmentation, and Depth

This tutorial explains a Qt-based C++ application that runs following four YOLO26s models on the same camera or video stream with a DEEPX NPU.
- YOLO26s Object Detection
- YOLO26s Pose Estimation
- YOLO26s Instance Segmentation
- YOLO26s Monocular Depth Estimation

![YOLO multi-channel demo](assets/yolo26-od-pos-seg-depth.png)

## Learning goals

By the end of this tutorial, you will be able to:

- understand the self-contained C++ project structure;
- identify the detection, pose, segmentation, and depth pipelines;
- understand how one input frame is distributed to four asynchronous workers;
- build the Qt application in Release mode; and
- run the demo with a camera or video file.

## 1. Locate the tutorial files

In [ ]:
from pathlib import Path
import shutil
import subprocess

candidates = [
    Path.cwd(),
    Path.cwd() / "notebooks" / "T21-demo-yolo26-od-pos-seg-depth",
]

TUTORIAL_ROOT = next(
    (path.resolve() for path in candidates if (path / "app" / "CMakeLists.txt").is_file()),
    None,
)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError("Could not find the T21-demo-yolo26-od-pos-seg-depth directory.")

APP_ROOT = TUTORIAL_ROOT / "app"
ASSET_ROOT = TUTORIAL_ROOT / "assets"

print(f"Tutorial: {TUTORIAL_ROOT}")
print(f"App:      {APP_ROOT}")
print(f"Assets:   {ASSET_ROOT}")

### Project layout

```text
T21-demo-yolo26-od-pos-seg-depth/
├── get_resources.sh
├── assets/
│   ├── models/
│   └── videos/
├── app/
│   ├── build.sh
│   ├── run_camera.sh
│   ├── run_video.sh
│   ├── CMakeLists.txt
│   ├── yolo26s_4.cpp
│   ├── common/
│   │   ├── base/
│   │   ├── processors/
│   │   └── utility/
│   ├── factory/
│   └── extern/
└── yolo26_od_pose_seg.ipynb
```


In [ ]:
required_files = [
    TUTORIAL_ROOT / "get_resources.sh",
    APP_ROOT / "build.sh",
    APP_ROOT / "run_camera.sh",
    APP_ROOT / "run_video.sh",
    APP_ROOT / "CMakeLists.txt",
    APP_ROOT / "yolo26s_4.cpp",
    APP_ROOT / "factory" / "yolo26s_factory.hpp",
    APP_ROOT / "factory" / "yolo26s_pose_factory.hpp",
    APP_ROOT / "factory" / "yolo26s_seg_factory.hpp",
]

for path in required_files:
    status = "OK" if path.is_file() else "MISSING"
    print(f"[{'OK' if path.is_file() else 'MISSING':7}] {path.relative_to(TUTORIAL_ROOT)}")

## 2. Check the environment

Install the Debian packages listed in `README.md`. The DEEPX device driver and DXRT SDK must also be installed.

In [ ]:
commands = ["g++", "cmake", "make", "qmake", "v4l2-ctl", "dxrt-cli"]
for command in commands:
    location = shutil.which(command)
    print(f"[{'OK' if location else 'MISSING':7}] {command}")

device_nodes = sorted(Path("/dev").glob("dxrt*"))
print(f"\nDEEPX device nodes: {device_nodes if device_nodes else 'not found'}")

## 3. Download and check the resources

`get_resources.sh` downloads the existing resource archive into `assets/` and removes the archive after successful extraction. The depth-model download will be added separately. Until then, place `yolo26-depth-s_768x768_q-lite.dxnn` under `assets/models/`.

The application expects these files by default:

```text
assets/
├── models/
│   ├── yolo26s.dxnn
│   ├── yolo26s-pose.dxnn
│   ├── yolo26s-seg.dxnn
│   └── yolo26-depth-s_768x768_q-lite.dxnn
└── videos/
    └── <input-video>
```

In [ ]:
default_assets = [
    ASSET_ROOT / "models" / "yolo26s.dxnn",
    ASSET_ROOT / "models" / "yolo26s-pose.dxnn",
    ASSET_ROOT / "models" / "yolo26s-seg.dxnn",
    ASSET_ROOT / "models" / "yolo26-depth-s_768x768_q-lite.dxnn",
]

for path in default_assets:
    print(f"[{'OK' if path.is_file() else 'MISSING':7}] {path.relative_to(TUTORIAL_ROOT)}")

video_files = sorted((ASSET_ROOT / "videos").glob("*"))
video_files = [path for path in video_files if path.is_file() and path.name != '.gitkeep']
print(f"\nAvailable video files: {len(video_files)}")
for path in video_files[:10]:
    print(f"  - {path.name}")

resources_missing = any(not path.is_file() for path in default_assets) or not video_files

Run the next cell only when resources are missing. Existing files with the same names may be replaced during extraction.

In [ ]:
if resources_missing:
    subprocess.run(
        [str(TUTORIAL_ROOT / "get_resources.sh")],
        cwd=TUTORIAL_ROOT,
        check=True,
    )
else:
    print("All required resources are already available.")

video_files = sorted((ASSET_ROOT / "videos").glob("*"))
video_files = [path for path in video_files if path.is_file() and path.name != '.gitkeep']
for path in default_assets:
    print(f"[{'OK' if path.is_file() else 'MISSING':7}] {path.relative_to(TUTORIAL_ROOT)}")
print(f"Available video files: {len(video_files)}")

missing_after_setup = [path for path in default_assets if not path.is_file()]
if missing_after_setup:
    missing_text = "\n".join(f"  - {path}" for path in missing_after_setup)
    raise FileNotFoundError(
        "Required model files are still missing after resource setup:\n" + missing_text
    )
if not video_files:
    raise FileNotFoundError("No input video was found under assets/videos.")

## 4. Application architecture

```text
Camera or video
       |
       v
CaptureThread
       |
       +--> Detection worker    --> Object Detection panel
       +--> Pose worker         --> Pose Estimation panel
       +--> Segmentation worker --> Instance Segmentation panel
       +--> Depth worker        --> Depth Estimation panel
```

The capture thread publishes each BGR frame to four latest-frame queues. Detection, pose, and segmentation use task-specific factories. `DepthWorker` performs 768 x 768 letterbox preprocessing, asynchronous DXRT inference, letterbox removal, resize to the source frame, and Turbo color mapping. Qt displays all four live results in a full-screen 2 x 2 grid.

## 5. Read the C++ code

The following helper displays selected sections from the current source files.

In [ ]:
from IPython.display import Code, display

def show_source(relative_path, marker, line_count=80):
    path = APP_ROOT / relative_path
    lines = path.read_text(encoding="utf-8").splitlines()
    try:
        start = next(index for index, line in enumerate(lines) if marker in line)
    except StopIteration as error:
        raise ValueError(f"Marker not found in {relative_path}: {marker}") from error

    end = min(start + line_count, len(lines))
    print(f"{relative_path}:{start + 1}-{end}")
    display(Code("\n".join(lines[start:end]), language="cpp"))

### 5.1. Build configuration

CMake builds one C++17 executable and links Qt5 Widgets, OpenCV, and DXRT. `PROJECT_ROOT_DIR` points to the tutorial directory so the default model paths resolve under `assets/models/`.

In [ ]:
show_source("CMakeLists.txt", "find_package(OpenCV", line_count=55)

### 5.2. Command-line options and default resources

`AppArgs` defines four model paths, camera settings, an optional video path, and debugging options. `--model-depth` overrides the default depth model. Omitting `--video` selects camera mode. Use `-c` or `--camera` to choose a V4L2 device and `--width`, `--height`, and `--fps` to request capture settings. If these options are omitted, the defaults are `/dev/video0`, 1280 x 720, and 30 FPS.

In [ ]:
show_source("yolo26s_4.cpp", "struct AppArgs", line_count=48)
show_source("yolo26s_4.cpp", "AppArgs parseArgs", line_count=70)

### 5.3. Task factories

Each factory creates the correct preprocessing, post-processing, and visualization components for one task.

In [ ]:
show_source("factory/yolo26s_factory.hpp", "class Yolo26sFactory", line_count=42)
show_source("factory/yolo26s_pose_factory.hpp", "class Yolo26s_poseFactory", line_count=42)
show_source("factory/yolo26s_seg_factory.hpp", "class Yolo26s_segFactory", line_count=42)

### 5.4. Asynchronous result workers

Each worker creates its own `InferenceEngine` and registers an asynchronous callback. Detection, pose, and segmentation use task-specific factories. The dedicated depth worker validates the `[1, 768, 768, 3]` UINT8 input and `[1, 1, 768, 768]` FLOAT output contract, preserves the input buffer until the callback completes, removes letterbox padding, restores the source-frame geometry, and applies OpenCV's Turbo color map. Each latest-frame queue replaces stale frames instead of building latency when a model is slower than the input stream.

In [ ]:
show_source("yolo26s_4.cpp", "void ResultWorker<ResultT, FactoryT>::run()", line_count=92)
show_source("yolo26s_4.cpp", "class DepthWorker final", line_count=82)
show_source("yolo26s_4.cpp", "void DepthWorker::preprocess", line_count=115)
show_source("yolo26s_4.cpp", "void DepthWorker::run()", line_count=125)

### 5.5. Camera and video capture

`CaptureThread` uses OpenCV `VideoCapture`. Video input loops unless `--no-loop-video` is set. Camera input uses the V4L2 backend and requests MJPG format.

In [ ]:
show_source("yolo26s_4.cpp", "void CaptureThread::runVideo()", line_count=65)
show_source("yolo26s_4.cpp", "void CaptureThread::runCamera()", line_count=65)

### 5.6. Qt 2 x 2 window

`QuadWindow` creates the four live panels, starts the four inference workers and capture thread, updates FPS labels, and performs an orderly shutdown.

In [ ]:
show_source("yolo26s_4.cpp", "class QuadWindow final", line_count=88)

## 6. Build the application

`build.sh` creates `app/build/`, configures CMake in Release mode, and runs `make` with all CPU cores reported by `nproc`.

In [ ]:
build_result = subprocess.run(
    ["./build.sh"],
    cwd=APP_ROOT,
    check=True,
)

binary_path = APP_ROOT / "build" / "yolo26s_4"
print(f"Build exit code: {build_result.returncode}")
print(f"Executable exists: {binary_path.is_file()}")

## 7. Run the camera demo

The default device is `/dev/video0` at a requested 1280 x 720 and 30 FPS. Change the variables below to select another camera or capture setting. The run cell opens a full-screen Qt window and blocks until the application exits.

In [ ]:
CAMERA_DEVICE = "/dev/video0"
CAMERA_WIDTH = 1280
CAMERA_HEIGHT = 720
CAMERA_FPS = 30

camera_path = Path(CAMERA_DEVICE)
print(f"Camera exists: {camera_path.exists()}")
if shutil.which("v4l2-ctl"):
    subprocess.run(["v4l2-ctl", "--list-devices"], check=False)

In [ ]:
if not binary_path.is_file():
    raise FileNotFoundError("Build the application before running the demo.")
if not camera_path.exists():
    raise FileNotFoundError(f"The camera is not available: {CAMERA_DEVICE}")

subprocess.run(
    [
        "./run_camera.sh",
        "--camera",
        CAMERA_DEVICE,
        "--width",
        str(CAMERA_WIDTH),
        "--height",
        str(CAMERA_HEIGHT),
        "--fps",
        str(CAMERA_FPS),
    ],
    cwd=APP_ROOT,
    check=True,
)

## 8. Run the video demo

Set `VIDEO_PATH` to a file under `assets/videos/`. `run_video.sh` accepts the video path as its first argument.

In [ ]:
VIDEO_PATH = video_files[0] if video_files else None
print(f"Selected video: {VIDEO_PATH if VIDEO_PATH else 'no video available'}")

In [ ]:
if not binary_path.is_file():
    raise FileNotFoundError("Build the application before running the demo.")
if VIDEO_PATH is None or not VIDEO_PATH.is_file():
    raise FileNotFoundError("Set VIDEO_PATH to an existing video file.")

subprocess.run(
    ["./run_video.sh", str(VIDEO_PATH)],
    cwd=APP_ROOT,
    check=True,
)

## Controls

- `Esc` or `q`: exit
- `EXIT` button: exit with the mouse

## Troubleshooting

- **Qt5 is not found:** install `qtbase5-dev` and run `./build.sh --clean`.
- **DXRT is not found:** verify the DXRT SDK installation and run `dxrt-cli -s`.
- **A model is missing:** place all four models under `assets/models/`, or pass explicit paths. The depth model defaults to `assets/models/yolo26-depth-s_768x768_q-lite.dxnn`.
- **The camera cannot be opened:** verify the V4L2 device and user permissions.
- **The window does not appear:** use a graphical desktop, remote desktop, or correctly configured X11 forwarding.
- **The notebook cell remains busy:** the run cell blocks while the Qt event loop is active; close the window to finish the cell.

## Summary

One capture thread sends each frame to four independent asynchronous DXRT pipelines. Task-specific factories handle object detection, pose estimation, and instance segmentation. A dedicated depth worker letterboxes the frame, runs the 768 x 768 depth model, restores the depth map to the source geometry, and applies a Turbo color map. Qt displays all four live results in a full-screen 2 x 2 layout.